# Brand and Social Radar, Culture Converts - Amend Parameters

## Initialisation

In [ ]:
from google.cloud import storage
# import google.generativeai as genai
import google.genai as genai
from google.cloud import bigquery
import os
import io
import time
import random
import json
import re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from google.api_core import exceptions
from google.cloud import secretmanager

In [ ]:
# Start the timer
start_time = time.time()

In [ ]:
def access_secret_version(project_id: str, secret_id: str, version_id: str = "latest") -> str:
    """
    Accesses the payload for the given secret version.
    """
    # Create the Secret Manager client.
    client = secretmanager.SecretManagerServiceClient()

    # Build the resource name of the secret version.
    # We use "latest" to always get the most current secret value.
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"

    try:
        # Access the secret version.
        response = client.access_secret_version(request={"name": name})

        # The payload is stored as bytes, so we decode it to a UTF-8 string.
        secret_value = response.payload.data.decode("UTF-8")

        return secret_value
    except Exception as e:
        print(f"Error accessing secret: {e}")
        return None

## Set up project and API Access

In [ ]:

PROJECT_ID = 'converged-brandradar-poc'
DATASET_ID = "brandradar"
REGION = 'europe-west2'
SECRET_ID = "GEMINI_API_KEY"

api_key = access_secret_version(PROJECT_ID, SECRET_ID)

if api_key:
    print("API Key retrieved successfully.")
    # Use the api_key variable in your application logic
    # external_api_call(api_key)
    client = genai.Client(api_key=api_key)
    # genai.configure(api_key=api_key)
else:
    print("Failed to retrieve API key.")


# model = genai.GenerativeModel(MODELNAME)
storage_client = storage.Client()


template_path_b = 'gs://converged-brandradar/templates/BrandRadar_Template.xlsx'
template_path_s = 'gs://converged-brandradar/templates/SocialRadar_Template.xlsx'
template_path_c = 'gs://converged-brandradar/templates/CultureConverts_template.xlsx'


API Key retrieved successfully.


## Parameters

In [ ]:
# Pick the model
# MODELNAME = 'models/gemini-2.5-flash'
MODELNAME = 'models/gemini-3.1-pro-preview'
# MODELNAME = 'models/gemini-3-pro-preview'
# MODELNAME = 'models/gemini-2.5-pro'
# If we choose BRAND and/or SOCIAL categories will be created
BRAND = True
SOCIAL = True
CULTURE = True

REPORTNAME = 'Moretti 230226 '
DATASOURCE='gs://converged-brandradar/lookups/Moretti.csv'


## Get Data

In [ ]:

# dfbrands = pd.read_csv('gs://converged-brandradar/lookups/allbrands.csv')
dfbrands = pd.read_csv(DATASOURCE)
# dfbrands = dfbrands.iloc[5:] # Drop a number of records if rerunning or splitting
dfbrands = dfbrands.head(1)  # Limit to first x rows for testing

# Create category totals
dfmarket_cats = dfbrands.groupby(['market', 'category']).size().reset_index()
dfmarket_cats['brand']='All Category'
print(dfmarket_cats.shape)

dfcult = dfbrands.copy()
if BRAND | SOCIAL:
    dfbrands = pd.concat([dfbrands, dfmarket_cats], ignore_index=True)

print(dfbrands.shape)

print(f"✓ Loaded {len(dfbrands)} brand records")
print(f"  Markets: {dfbrands['market'].unique().tolist()}")
print(f"  Categories: {dfbrands['category'].unique().tolist()}")



(1, 4)
(2, 4)
✓ Loaded 2 brand records
  Markets: ['UK']
  Categories: ['Beer']


In [ ]:
dfbrands

,brand,category,market,0
0,Madri Excepcional,Beer,UK,NaN
1,All Category,Beer,UK,1.0


In [ ]:
# Do we just need to select certain categories or markets?
# dfbrands = dfbrands[dfbrands['category'].isin(['Drugs and Pharma (OTC products)','Supermarket & Grocery stores','Non Alcoholic Drinks - soft drinks'])]
# dfbrands = dfbrands[dfbrands['category'].isin(['Content distributors & producers - Media, platforms and networks'])]
# dfbrands = dfbrands[dfbrands['market'].isin(['UK'])]


## Functions

In [ ]:
def extract_json_robust(response_text):
    """
    Robust JSON extraction with multiple fallback strategies.
    Returns parsed JSON dict or None if all strategies fail.
    """
    if not response_text:
        return None

    # Strategy 1: JSON in markdown code block
    json_match = re.search(r'```json\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # Strategy 2: Any JSON object in the text
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(0))
        except json.JSONDecodeError:
            pass

    # Strategy 3: Entire response is JSON
    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        pass

    return None

In [ ]:
def validate_json_fields(json_data, required_fields):
    """
    Validate that JSON contains all required fields.
    Returns (is_valid, missing_fields_list)
    """
    if json_data is None:
        return False, required_fields

    missing = [field for field in required_fields if field not in json_data]
    return len(missing) == 0, missing

In [ ]:
def safe_get(data, key, default=None):
    """Safely get value from dict with default fallback"""
    if isinstance(data, dict):
        return data.get(key, default)
    return default

In [ ]:
# --- The Decorator
def retry_with_backoff(retries=5, backoff_in_seconds=1):
    def rwb(f):
        def wrapper(*args, **kwargs):
            _retries, _backoff = retries, backoff_in_seconds
            while _retries > 1:
                try:
                    return f(*args, **kwargs)
                except exceptions.ResourceExhausted as e:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"API quota exceeded. Retrying in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
                except Exception as e:
                    print(f"An unexpected error occurred: {e}. Retrying...")
                    time.sleep(_backoff)
                    _retries -= 1
            return f(*args, **kwargs)
        return wrapper
    return rwb

In [ ]:
def setbranddf():
    df = pd.DataFrame(columns=["Brand", "Category", "Market", "Summary", "mentions", "mentions_description","overallimp", "overallimp_description", "purchase", "purchase_description",
                           "premium", "premium_description","sustainable", "sustainable_description",
                           "trust","trust_description","recommend","recommend_description","dynamic", "dynamic_description", "functional", "functional_description",
                           "personal", "personal_description","collective", "collective_description",
                           "touchpoints_description", "channel_description", "journey_description", "branded_description", "sponsorship_description", "cgc_description","brand_generated_description","date"])
    df['mentions'] = df['mentions'].astype('int64')
    df['overallimp'] = df['overallimp'].astype('int64')
    df['purchase'] = df['purchase'].astype('int64')
    df['premium'] = df['premium'].astype('int64')
    df['sustainable'] = df['sustainable'].astype('int64')
    df['trust'] = df['trust'].astype('int64')
    df['recommend'] = df['recommend'].astype('int64')
    df['dynamic'] = df['dynamic'].astype('int64')
    df['functional'] = df['functional'].astype('int64')
    df['personal'] = df['personal'].astype('int64')
    df['collective'] = df['collective'].astype('int64')
    df['date'] = df['date'].astype('datetime64[ns]')
    return(df)




In [ ]:
def setsocialdf():
    df = pd.DataFrame(columns=["Brand", "Category", "Market", "Summary", "sentiment", "sentiment_description","emotion", "emotion_description","positivity", "positivity_description",
                           "shift","shift_description","spike","spike_description","themes", "themes_description", "emerging_topics", "emerging_topics_description",
                           "debates", "debates_description","platforms", "platforms_description","engagement", "engagement_description","media_type", "media_type_description","date"])
    df['sentiment'] = df['sentiment'].astype('int64')
    df['emotion'] = df['emotion'].astype('int64')
    df['positivity'] = df['positivity'].astype('int64')
    df['shift'] = df['shift'].astype('int64')
    df['spike'] = df['spike'].astype('int64')
    df['themes'] = df['themes'].astype('int64')
    df['emerging_topics'] = df['emerging_topics'].astype('int64')
    df['debates'] = df['debates'].astype('int64')
    df['platforms'] = df['platforms'].astype('int64')
    df['engagement'] = df['engagement'].astype('int64')
    df['media_type'] = df['media_type'].astype('int64')
    df['date'] = df['date'].astype('datetime64[ns]')
    return(df)




In [ ]:
def setculturedf():
    """Initialize empty DataFrame for brand results"""
    df = pd.DataFrame(columns=[
        "Brand", "Category", "Market",
        "Buzz1", "Buzz2", "Buzz3", "Buzz4", "Buzz5","Belong1", "Belong2", "Belong3", "Belong4", "Belong5",
        "Belief1", "Belief2", "Belief3", "Belief4", "Belief5", "Behave1", "Behave2", "Behave3", "Behave4", "Behave5",
        "date", "projectname"
    ])

    # Set data types
    int_cols = ["Buzz1", "Buzz2", "Buzz3", "Buzz4", "Buzz5","Belong1", "Belong2", "Belong3", "Belong4", "Belong5",
        "Belief1", "Belief2", "Belief3", "Belief4", "Belief5", "Behave1", "Behave2", "Behave3", "Behave4", "Behave5"]
    for col in int_cols:
        df[col] = df[col].astype('int64')
    df['date'] = df['date'].astype('datetime64[ns]')

    return df

In [ ]:
def set_questionbrand(brand, category, market):
    return f"""Please provide a brief (approximately 200 words) summary of the {brand} brand's perception in the {category} category in the {market} market.

        For each of the following aspects, provide a score out of a 100 with a very short description of the score:
        - How often is it mentioned?
        - What is your overall impression of the brand?
        - Have people purchased the brand?
        - Is it thought of as a premium brand that people would pay a premium price for?
        - It is considered a sustainable brand?
        - do people trust it?
        - Would they recommend it to others?
        - What do you think of it's speed, dynamism and willingness to adapt to change?
        - Does the brand meet functional requirements, for example, seamless digital, in-store or purchasing experience, product range, highly innovative,
        value for money, offers price consistency, time saving, high quality products, respectful of customers and their data, offers exclusive experiences,
        offers unique products and services, safe products and services, acts like a leader or challenger brand, has a good reputation, delivers on what it says?        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing, lets me escape from the everyday, inspires me with new opportunities and ideas,
        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing,
        lets me escape from the everyday, inspires me with new opportunities and ideas, instilling peace of mind and confidence, connecting people,
        enables me to be smart with money and time, simplifies my life, helps me express myself as an individual, feel more attractive and stylish,
        make me feel special and unique, makes me energised and alive, helps me feel in control of my life, gives me a sense of happiness, makes me feel good,
        makes me feel like it is looking after my best interests?
         - Does the brand meet collective requirements, for example does it act with integrity, is it ethical, is it transparent in its operations,
         is it considered a good employer, offers sustainable products, it fights against poverty and hunger, supports healthy living and promotes well-being,
         supports culture and education, supports equality, diversity and inclusion, supports social issues and good causes,
         contributes to the national economy and my local community, invests in innovative, sustainable and ethical solutions, inspires sustainable,
         responsible behaviours and consumption, committed to making products/services more sustainable?

        and can you provide short descriptions of the following questions
        - What are the most common touchpoints associated with the brand?
        - Which channel do consumers mention most when discussing the brand?
        - Where has the brand appeared most in the consumer journey online
        - What formats of branded content does brand most often use?
        - What recent sponsorships or partnerships has the brand activated?
        - What kinds of consumer-generated content appear in discussions about the brand?
        - What kinds of brand-generated content appear in discussions about the brand?



        Could the results be returned in JSON format with the brand in a variable called brand, the country in a field called market, the summary in a field called summary,
        the individual scores and the short descriptions of the scores in fields called, mentions, mentions_description, overallimp, overallimp_description, purchase, purchase_description, premium, premium_description,
        sustainable, sustainable_description, trust, trust_description, recommend, recommend_description, dynamic, dynamic_description, functional, functional_description, personal, personal_description, collective, collective_description,
        touchpoints_description, channel_description, journey_description, branded_description, sponsorship_description, cgc_description, brand_generated_description?

"""

In [ ]:
def set_questionsocial(brand, category, market):
    return  f"""Please provide a brief (approximately 200 words) summary of the {brand} brand’s public sentiment and online conversation dynamics in the {category} category in the {market} market based on recent digital signals.

 	For each of the following aspects, provide a score out of 100 and a short description explaining the score:
	- What is the general sentiment about the brand?
	- What emotions are most commonly associated with the brand?
	- Is the brand currently viewed more positively overall?
	- Is the brand currently viewed more negatively overall?
	- Are there noticeable sentiment shifts over the last 6 months?
	- Have there been sentiment spikes triggered by campaigns, news, or events?
	- What are the most discussed themes in online conversations about the brand?
	- What social, cultural, product or service topics are emerging?
	- Are there any polarizing or divisive conversations surrounding the brand?
	- Which platforms (e.g. Reddit, X, TikTok, forums, blogs) host the most conversations about the brand?
	- Where does the brand get the most engagement?
	- Is brand visibility driven more by owned channels or third-party/earned media?

 Please return results in **JSON format** the using the following fields:
- `"brand"` for the brand name
- `"market"` for the country
- `"summary"` for the narrative overview
- `"scores"` as an object with:
  - `sentiment`, `emotion`, `positivity`, `sentiment_shift`, `sentiment_spike`
  - `themes`, `emerging_topics`, `debates`
  - `platforms`, `engagement`, `media_type`

Each object should contain:
- `score` (0–100)
- `description` (1–2 sentence explanation)
"""

In [ ]:
def set_questionculture(brand, category, market):
    """Generate brand analysis prompt with strict JSON requirements"""
    return f"""Please analyze the {brand} brand's perception in the {category} category in the {market} market.
I am interested in understanding 4 areas, buzz, belonging, belief and behaviour.  Could provide a value between 1 and 100 for the following questions and store them in
JSON format with fields called Buzz1 etc?
Buzz1: Measuring attention. Number of mentions (total #) - across social, news, forums.
Buzz2: Share of cultural voice. % of culturally relevant mentions vs peer brands in the target market.
Buzz3:  Competitor share. Share of voice vs competitors (%) - brand mentions as a % of category mentions.
Buzz4: Authority-weighted presence. Weighted score based on coverage in high-authority media and top creators.
Buzz5: Topic spread. Count of distinct cultural themes (e.g., sustainability, arts, finance) where the brand appears organically.
Belong1: Cultural fit score. % of contextual mentions coded as “authentic” vs “performative”.
Belong2: Community acceptance index. Proportion of mentions coming from identified target communities/subcultures (%).
Belong3: Peer comparison on affinity. Relative score vs. category peers for perceived cultural fit.
Belong4: Audience inclusion signal. Frequency of language indicating inclusion (e.g., “for us,” “represents,” “part of”, “this is for people like”).
Belong5: Negative fit incidence. % of mentions expressing rejection or misalignment with cultural spaces.
Belief1: Cultural authority score. % of mentions framing the brand as a leader, innovator, or credible voice in cultural or social topics.
Belief2: Value alignment index. Frequency of positive associations with values (e.g., sustainability, diversity, ethics) in cultural discourse.
Belief3: Thought leadership presence. Number of named endorsements or quotes from recognised cultural leaders/experts.
Belief4: Peer Comparison on Authority. Relative score vs category peers for perceived leadership and credibility.
Belief5: Negative Authority Incidence. % of mentions questioning or criticizing the brand’s intent or values.
Behave1: Action conversation signals. Frequency of mentions indicating concrete action (e.g., “I attended”, “I bought”, “I downloaded”, purchase, trial, sign-up).
Behave2: Advocacy rate. % of user-generated content showing active recommendation or endorsement of the brand.
Behave3: Cultural participation index. Number of instances where audiences join brand-led cultural initiatives (e.g., events, challenges, collaborations).
Behave4: Behavioural shift mentions. Mentions where users report changing habits or choices influenced by the brand’s cultural activity.
Behave5: Peer comparison on adoption. Relative score vs category peers for observable cultural-driven actions.

CRITICAL: Return ONLY a valid JSON object with NO markdown formatting, NO code blocks, NO additional text.

Required JSON structure:
{{
  "brand": "{brand}",
  "market": "{market}",
  "Buzz1": integer,
  "Buzz2": integer,
  "Buzz3": integer,
  "Buzz4": integer,
  "Buzz5": integer,
  "Belong1": integer,
  "Belong2": integer,
  "Belong3": integer,
  "Belong4": integer,
  "Belong5": integer,
  "Belief1": integer,
  "Belief2": integer,
  "Belief3": integer,
  "Belief4": integer,
  "Belief5": integer,
  "Behave1": integer,
  "Behave2": integer,
  "Behave3": integer,
  "Behave4": integer,
  "Behave5": integer
}}"""

In [ ]:
def add_to_dataframebrand(df, category, market, response_text):
    # Extract relevant data from the response and add it to the DataFrame
    json_match = re.search(r'```json\s*(\{.*\})\s*```', response_text, re.DOTALL)
    if json_match:
        brand_json_str = json_match.group(1)
        brand_json = json.loads(brand_json_str)
    else:
        print("No JSON object found in gemini_text_response.")

    # Create a new row for the DataFrame
    try:
        new_row = {
            "Brand": brand_json['brand'],
            "Category": category,
            "Market": market,
            "Summary": brand_json['summary'],
            "mentions": brand_json['mentions'],
            "mentions_description": brand_json['mentions_description'],
            "overallimp": brand_json['overallimp'],
            "overallimp_description": brand_json['overallimp_description'],
            "purchase": brand_json['purchase'],
            "purchase_description": brand_json['purchase_description'],
            "premium": brand_json['premium'],
            "premium_description": brand_json['premium_description'],
            "sustainable": brand_json['sustainable'],
            "sustainable_description": brand_json['sustainable_description'],
            "trust": brand_json['trust'],
            "trust_description": brand_json['trust_description'],
            "recommend": brand_json['recommend'],
            "recommend_description": brand_json['recommend_description'],
            "dynamic": brand_json['dynamic'],
            "dynamic_description": brand_json['dynamic_description'],
            "functional": brand_json['functional'],
            "functional_description": brand_json['functional_description'],
            "personal": brand_json['personal'],
            "personal_description": brand_json['personal_description'],
            "collective": brand_json['collective'],
            "collective_description": brand_json['collective_description'],
            "touchpoints_description": brand_json['touchpoints_description'],
            "channel_description": brand_json['channel_description'],
            "journey_description": brand_json['journey_description'],
            "branded_description": brand_json['branded_description'],
            "sponsorship_description": brand_json['sponsorship_description'],
            "cgc_description": brand_json['cgc_description'],
            "brand_generated_description": brand_json['brand_generated_description'],
            "date": datetime.now().strftime("%Y/%m/%d")
        }

    # Append the new row to the DataFrame
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    except Exception as e:
        print("Error in add_to_dataframesocial:",e)
    return df



In [ ]:
def add_to_dataframesocial(df, category, market, response_text):
     # Extract relevant data from the response and add it to the DataFrame
    json_match = re.search(r'```json\s*(\{.*\})\s*```', response_text, re.DOTALL)
    if json_match:
        brand_json_str = json_match.group(1)
        brand_json = json.loads(brand_json_str)
    else:
        print("No JSON object found in gemini_text_response.")
    # print(brand_json['brand'])
    # print(brand_json['market'])
    # print(brand_json['scores'])
    # Create a new row for the DataFrame
    try:
        new_row = {
            "Brand": brand_json['brand'],
            "Category": category,
            "Market": market,
            "Summary": brand_json['summary'],
            "sentiment": brand_json['scores']['sentiment']['score'],
            "sentiment_description": brand_json['scores']['sentiment']['description'],
            "emotion": brand_json['scores']['emotion']['score'],
            "emotion_description": brand_json['scores']['emotion']['description'],
            "positivity": brand_json['scores']['positivity']['score'],
            "positivity_description": brand_json['scores']['positivity']['description'],
            "shift": brand_json['scores']['sentiment_shift']['score'],
            "shift_description": brand_json['scores']['sentiment_shift']['description'],
            "spike": brand_json['scores']['sentiment_spike']['score'],
            "spike_description": brand_json['scores']['sentiment_spike']['description'],
            "themes": brand_json['scores']['themes']['score'],
            "themes_description": brand_json['scores']['themes']['description'],
            "emerging_topics": brand_json['scores']['emerging_topics']['score'],
            "emerging_topics_description": brand_json['scores']['emerging_topics']['description'],
            "debates": brand_json['scores']['debates']['score'],
            "debates_description": brand_json['scores']['debates']['description'],
            "platforms": brand_json['scores']['platforms']['score'],
            "platforms_description": brand_json['scores']['platforms']['description'],
            "engagement": brand_json['scores']['engagement']['score'],
            "engagement_description": brand_json['scores']['engagement']['description'],
            "media_type": brand_json['scores']['media_type']['score'],
            "media_type_description": brand_json['scores']['media_type']['description'],
            "date": datetime.now().strftime("%Y/%m/%d")
        }

        # Append the new row to the DataFrame
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    except Exception as e:
        print("Error in add_to_dataframesocial:",e)
    return df

In [ ]:
def add_to_dataframeculture(df, brand_name, category, market, response_text):
    """
    Add brand analysis results to DataFrame with robust error handling.
    """
    brand_json = extract_json_robust(response_text)

    if brand_json is None:
        print(f"  ❌ Failed to extract JSON for {brand_name} in {market}")
        # Create error row
        new_row = {
            "Brand": brand_name,
            "Category": category,
            "Market": market,
            "Buzz1": "ERROR: Failed to parse API response",
            "Buzz2": "Parse Error",
            "Buzz3": "Parse Error",
            "Buzz4": "Parse Error",
            "Buzz5": "Parse Error",
            "Belong1": "Parse Error",
            "Belong2": "Parse Error",
            "Belong3": "Parse Error",
            "Belong4": "Parse Error",
            "Belong5": "Parse Error",
            "Belief1": "Parse Error",
            "Belief2": "Parse Error",
            "Belief3": "Parse Error",
            "Belief4": "Parse Error",
            "Belief5": "Parse Error",
            "Behave1": "Parse Error",
            "Behave2": "Parse Error",
            "Behave3": "Parse Error",
            "Behave4": "Parse Error",
            "Behave5": "Parse Error",
            "date": datetime.now().strftime("%Y/%m/%d"),
            "projectname": REPORTNAME
        }
        return pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    try:
        new_row = {
            "Brand": safe_get(brand_json, 'brand', brand_name),
            "Category": category,
            "Market": market,
            "Buzz1": safe_get(brand_json, 'Buzz1', ''),
            "Buzz2": safe_get(brand_json, 'Buzz2', ''),
            "Buzz3": safe_get(brand_json, 'Buzz3', ''),
            "Buzz4": safe_get(brand_json, 'Buzz4', ''),
            "Buzz5": safe_get(brand_json, 'Buzz5', ''),
            "Belong1": safe_get(brand_json, 'Belong1',''),
            "Belong2": safe_get(brand_json, 'Belong2', ''),
            "Belong3": safe_get(brand_json, 'Belong3', ''),
            "Belong4": safe_get(brand_json, 'Belong4', ''),
            "Belong5": safe_get(brand_json, 'Belong5', ''),
            "Belief1": safe_get(brand_json, 'Belief1', ''),
            "Belief2": safe_get(brand_json, 'Belief2', ''),
            "Belief3": safe_get(brand_json, 'Belief3', ''),
            "Belief4": safe_get(brand_json, 'Belief4', ''),
            "Belief5": safe_get(brand_json, 'Belief5', ''),
            "Behave1": safe_get(brand_json, 'Behave1', ''),
            "Behave2": safe_get(brand_json, 'Behave2', ''),
            "Behave3": safe_get(brand_json, 'Behave3', ''),
            "Behave4": safe_get(brand_json, 'Behave4', ''),
            "Behave5": safe_get(brand_json, 'Behave5', ''),
            "date": datetime.now().strftime("%Y/%m/%d"),
            "projectname": REPORTNAME
        }

        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        print(f"  ✓ Successfully processed")

    except Exception as e:
        print(f"  ❌ Error creating row: {e}")

    return df


In [ ]:
@retry_with_backoff(retries=3, backoff_in_seconds=2)
def runQuery(question):
    """
    Generates content from the Gemini model with retries for transient errors.

    Args:
        question: The prompt to send to the model.

    Returns:
        The generated text content as a string, or None if the request
        was blocked or failed permanently.
    """
    try:
        response = client.models.generate_content(model=MODELNAME, contents=question)
        # After a successful call, specifically check for a blocked response
        if not response.candidates:
            print(f"Request was blocked. Reason: {response.prompt_feedback}")
            return None

        # The .text property is the safest way to get the text content.
        # It raises a ValueError if the response is blocked, which we can catch.
        return response.text

    except ValueError:
        # This catches the error from .text if the response was malformed or empty
        print("Response was empty or blocked after a successful API call.")
        print(f"Safety Feedback: {response.prompt_feedback}")
        return None


In [ ]:
def runbrands(dfbrands,df):
    for idx, row in dfbrands.iterrows():
        question = set_questionbrand(row['brand'],row['category'],row['market'])
        category=row['category']
        market=row['market']
        print('Processing Brand ', row['brand'],row['market'])  # Print brand and market for debugging
        gemini_text_response = runQuery(question)
        if not gemini_text_response:
            gemini_text_response = "Error: Gemini returned an empty response or no text content found."
        # Add to DataFrame
        df = add_to_dataframebrand(df, category, market, gemini_text_response)
        # print(f"Gemini's response received.")
        response_data = {
            "timestamp": datetime.now().isoformat(),
            "input_question": question,
            "category":category,
            "gemini_response": gemini_text_response
        }
        # Store the basic JSON response data
        all_response_data.append(response_data)
    return(all_response_data,df)



In [ ]:
def runsocial(dfbrands,df):
    for idx, row in dfbrands.iterrows():
        question = set_questionsocial(row['brand'],row['category'],row['market'])
        category=row['category']
        market=row['market']
        print('Processing Social ', row['brand'],row['market'])  # Print brand and market for debugging
        gemini_text_response = runQuery(question)
        if not gemini_text_response:
            gemini_text_response = "Error: Gemini returned an empty response or no text content found."
            # print("Social - ",row['brand'],row['market'], ' Failed')
        # Add to DataFrame
        df = add_to_dataframesocial(df, category, market, gemini_text_response)
        response_data = {
            "timestamp": datetime.now().isoformat(),
            "input_question": question,
            "category":category,
            "gemini_response": gemini_text_response
        }
        # Store the basic JSON response data
        all_response_data.append(response_data)

    return(all_response_data,df)

In [ ]:
def runculture(dfbrands, df, start_idx=0):
    """
    Process brand analysis for all records with progress tracking.
    """
    all_response_data = []
    total = len(dfbrands)

    failed_records = []

    print(f"\n{'='*60}")
    print(f"CULTURAL ANALYSIS - Processing {total} records")
    print(f"{'='*60}\n")

    for idx, row in dfbrands.iterrows():
        if idx < start_idx:
            continue

        progress = f"[{idx+1}/{total}]"
        print(f"{progress} Processing: {row['brand']} - {row['market']}")

        try:
            question = set_questionculture(row['brand'], row['category'], row['market'])
            gemini_text_response = runQuery(question)

            if not gemini_text_response:
                print(f"  ❌ Failed to get response from API")
                gemini_text_response = '{"error": "No response received"}'
                failed_records.append({
                    'index': idx,
                    'brand': row['brand'],
                    'market': row['market'],
                    'error': 'No API response'
                })

            # Add to DataFrame
            df = add_to_dataframeculture(
                df, row['brand'], row['category'], row['market'], gemini_text_response
            )

            # Store response data
            response_data = {
                "timestamp": datetime.now().isoformat(),
                "brand": row['brand'],
                "category": row['category'],
                "market": row['market'],
                "input_question": question,
                "gemini_response": gemini_text_response,
                "success": extract_json_robust(gemini_text_response) is not None
            }
            all_response_data.append(response_data)

            # Checkpoint every 10 records
            if (idx + 1) % 10 == 0:
                print(f"  💾 Checkpoint: {idx+1} records processed")

            # Rate limiting
            time.sleep(1)

        except Exception as e:
            print(f"  ❌ CRITICAL ERROR: {e}")
            failed_records.append({
                'index': idx,
                'brand': row['brand'],
                'market': row['market'],
                'error': str(e)
            })
            continue

    # Report failed records
    if failed_records:
        print(f"\n⚠️  {len(failed_records)} records failed:")
        for rec in failed_records:
            print(f"  - {rec['brand']} ({rec['market']}): {rec['error']}")

    return all_response_data, df


In [ ]:
def writejson(all_response_data,repname):
  bucket_name = "converged-brandradar"
  destination_blob_name = f"results/{repname} {datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

  try:
      storage_client = storage.Client()
      bucket = storage_client.bucket(bucket_name)
      blob = bucket.blob(destination_blob_name)
      # We use json.dumps() here instead of json.dump()
      blob.upload_from_string(
          data=json.dumps(all_response_data, indent=4, ensure_ascii=False),
          content_type='application/json'  # Set the content type for proper handling
      )

      print(f"\nResponse successfully saved to 'gs://{bucket_name}/{destination_blob_name}'")

  except Exception as e:
      print(f"Error saving data to GCS: {e}")


In [ ]:
def writexls(df, repname, report_name, template_path, storage_client):
    # Define the GCS output path
    outname_gcs_path = f"gs://converged-brandradar/results/{repname} {report_name} {datetime.now().strftime('%Y%m%d')}.xlsx"

    try:
        # --- 1. Download template from GCS into an in-memory buffer ---
        bucket_name = template_path.split('/')[2]
        blob_path = '/'.join(template_path.split('/')[3:])

        bucket = storage_client.bucket(bucket_name)
        blob = bucket.blob(blob_path)

        in_memory_template = io.BytesIO()
        blob.download_to_file(in_memory_template)
        in_memory_template.seek(0) # Go to the beginning of the buffer
        print(f"Template '{template_path}' downloaded into memory.")

        # --- 2. Load and Modify Workbook from memory ---
        book = load_workbook(in_memory_template)

        # Update the 'Key' sheet
        sheet = book['Key']
        sheet.cell(row=3, column=1, value=report_name)

        # Select the correct sheet for the data dump
        if repname == 'Brand':
            sheet = book['BrandRadar Results']
        elif repname == 'Social':
            sheet = book['SocialRadar Results']
        elif repname == 'Cultural Converts':
            sheet = book['Cultural Converts']
        else:
            raise ValueError(f"Unknown report name '{repname}'. Cannot find a target sheet.")

        # Write the DataFrame to the selected sheet
        rows = dataframe_to_rows(df, index=False, header=False)
        for r_idx, row in enumerate(rows, 2):  # Start from row 2
            for c_idx, value in enumerate(row, 1):
                sheet.cell(row=r_idx, column=c_idx, value=value)

        # --- 3. Save the modified workbook to an in-memory buffer ---
        in_memory_output = io.BytesIO()
        book.save(in_memory_output)
        in_memory_output.seek(0)

        # --- 4. Upload the in-memory buffer to GCS ---
        out_bucket_name = outname_gcs_path.split('/')[2]
        out_blob_path = '/'.join(outname_gcs_path.split('/')[3:])

        out_bucket = storage_client.bucket(out_bucket_name)
        out_blob = out_bucket.blob(out_blob_path)
        out_blob.upload_from_file(in_memory_output, content_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

        print(f"Report successfully created and uploaded to '{outname_gcs_path}'")

    except Exception as e:
        print(f"An error occurred: {e}")


In [ ]:
def append_to_bigquery(df, project_id, dataset_id, table_id,schema):

  try:
    client = bigquery.Client(project=project_id)

    table_ref = f"{project_id}.{dataset_id}.{table_id}"

    try:
      # Check if the table exists
      client.get_table(table_ref)
      job_config = bigquery.LoadJobConfig(
          write_disposition=bigquery.WriteDisposition.WRITE_APPEND  # Append if exists
      )
      print(f"Table {table_id} exists. Appending data.")

    except Exception as e:
      if "Not found" in str(e):  # Table doesn't exist
        job_config = bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # Create if doesn't exist
            schema=schema
        )
        print(f"Table {table_id} does not exist. Creating and loading data.")

      else:  # Other BigQuery error
        print(f"An error occurred checking table existence: {e}")
        return

    job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
    job.result()  # Wait for the job to complete

    print(f"Successfully loaded {len(df)} rows to BigQuery table {table_id}")

  except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
def convert_schema(df):
    schema = []
    for col in df.columns:
      if df[col].dtype == 'int64':
        schema.append(bigquery.SchemaField(col, "INTEGER"))
      elif df[col].dtype == 'float64':
        schema.append(bigquery.SchemaField(col, "FLOAT"))
      elif df[col].dtype == 'bool':
        schema.append(bigquery.SchemaField(col, "BOOLEAN"))
      elif df[col].dtype == 'datetime64[ns]':  # Check for datetime type
        schema.append(bigquery.SchemaField(col, "TIMESTAMP")) # Use appropriate BQ type
      else:
        schema.append(bigquery.SchemaField(col, "STRING"))
    return(schema)

## Control

In [ ]:
# Set up structure for storing responses
all_response_data=[]
project_id = "converged-brandradar-poc"
dataset_id = "brandradar"
if BRAND:
    all_response_data=[]
    df = setbranddf()
    all_response_data, df = runbrands(dfbrands,df)
    writejson(all_response_data, "Brand")
    storage_client = storage.Client()
    writexls(df, "Brand", REPORTNAME, template_path_b, storage_client)
    df['projectname'] =  REPORTNAME
    schema=convert_schema(df)
    table_id = "BrandResults"
    append_to_bigquery(df, project_id, dataset_id, table_id,schema)
    #print(df.head())
    print("Brand Report Complete")
    print("---------------------")
if SOCIAL:
    all_response_data=[]
    df = setsocialdf()
    all_response_data, df = runsocial(dfbrands,df)
    writejson(all_response_data, 'Social')
    storage_client = storage.Client()
    writexls(df, "Social", REPORTNAME, template_path_s, storage_client)
    df['projectname'] =  REPORTNAME
    table_id = "SocialResults"
    schema=convert_schema(df)
    append_to_bigquery(df, project_id, dataset_id, table_id,schema)
    #print(df.head())
    print("Social Report Complete")
    print("----------------------")
if CULTURE:
    all_response_data=[]
    df = setculturedf()
    # Filter dfbrands for culture report: exclude brands starting with 'All'
    all_response_data, df = runculture(dfcult,df)
    print(df.head())
    writejson(all_response_data, 'Culture')
    storage_client = storage.Client()
    writexls(df, "Cultural Converts", REPORTNAME, template_path_c, storage_client)
    df['projectname'] =  REPORTNAME
    table_id = "CultureConverts"
    schema=convert_schema(df)
    append_to_bigquery(df, project_id, dataset_id, table_id,schema)
    #print(df.head())
    print("Culture Converts Report Complete")
    print("----------------------")
print("Process Complete")

Processing Brand  Madri Excepcional UK
Processing Brand  All Category UK

Response successfully saved to 'gs://converged-brandradar/results/Brand 20260223_154850.json'
Template 'gs://converged-brandradar/templates/BrandRadar_Template.xlsx' downloaded into memory.
Report successfully created and uploaded to 'gs://converged-brandradar/results/Brand Moretti 230226  20260223.xlsx'
Table BrandResults exists. Appending data.
Successfully loaded 2 rows to BigQuery table BrandResults
Brand Report Complete
---------------------
Processing Social  Madri Excepcional UK
Processing Social  All Category UK

Response successfully saved to 'gs://converged-brandradar/results/Social 20260223_155037.json'
Template 'gs://converged-brandradar/templates/SocialRadar_Template.xlsx' downloaded into memory.
Report successfully created and uploaded to 'gs://converged-brandradar/results/Social Moretti 230226  20260223.xlsx'
Table SocialResults exists. Appending data.
Successfully loaded 2 rows to BigQuery table S

In [ ]:
# df.head()

In [ ]:
end_time = time.time()
print(f"Run time: {end_time - start_time:.2f} seconds")

Run time: 264.65 seconds


In [ ]:
# print(df.info())

## Check what models are available if required

Dataset details:
project_id = "Converged BrandRadar POC"
dataset_id = "brandradar"
table_id = "SocialResults"
table_id = "BrandResults"

In [ ]:
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-001
models/imagen-4.0-ultra-generate-001
models/imagen-4.0-fast

This code iterates through all available models associated with your `client` object and prints their names.